# 05 · Contextual aggregator: one final answer from the five agents

Runtime > Change runtime type > **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import getpass, os
GITHUB_USER = 'YOUR_GITHUB_USERNAME'
REPO = 'YOUR_REPO_NAME'
token = getpass.getpass('GitHub token (leave blank if repo is public): ')
auth = f'{token}@' if token else ''
%cd /content
!rm -rf /content/repo
!git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
del token, auth

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

In [ ]:
import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/NLP-Final'
os.environ['SCRATCH_DIR'] = '/content/scratch'

### 1. Build the data and let all ten agents answer the training questions
About an hour. Resumable: if Colab disconnects, rerun the setup cells and this cell; finished agents are skipped.

In [ ]:
!python src/prepare_aggregator_data.py

### 2. Train one aggregator per arm (identical settings)

In [ ]:
!python src/train_aggregator.py --arm lora
!python src/train_aggregator.py --arm full

### 3. Evaluate every system on the held-out questions
The base model and the selection rule need no generation; the prompted and trained aggregators do.

In [ ]:
!python src/eval_aggregator.py --system pt
for arm in ['full', 'lora']:
    !python src/eval_aggregator.py --system select --arm {arm}
    !python src/eval_aggregator.py --system prompted --arm {arm}
    !python src/eval_aggregator.py --system agg --arm {arm}

### 4. Compare

In [ ]:
import json, pandas as pd
R = os.environ['PROJECT_ROOT'] + '/results'
systems = ['pt', 'select_full', 'select_lora', 'prompted_full', 'prompted_lora', 'agg_full', 'agg_lora']
rows = {s: json.load(open(f'{R}/final_{s}.json')) for s in systems}
pd.DataFrame({s: {k: r[k] for k in ['n', 'rouge1', 'rouge2', 'rougeL', 'rougeLsum', 'profile_reward']} for s, r in rows.items()}).T

In [ ]:
# A few final answers side by side
pd.set_option('display.max_colwidth', 250)
for s in ['pt', 'agg_lora']:
    print(s, pd.read_csv(f'{R}/final_{s}.csv')['reply'].head(3).tolist(), '\n')
print('reference', pd.read_csv(f'{R}/final_pt.csv')['reference'].head(3).tolist())